# Integración histórica de los Conteos Censales 2009--2024

**Proyecto:** Hackea la participación: Datos que construyen democracia  
**Reto 1:** Perfiles ciudadanos de participación  
**Etapa:** Integración histórica previa al clustering

## Objetivo

Integrar las seis bases procesadas de 2009, 2012, 2015, 2018, 2021 y 2024 utilizando como llave común:

$$
\text{Entidad}+\text{Grupo de edad}+\text{Sexo}+\text{Tipo de sección}
$$

El notebook genera:

- una **base histórica maestra**, que conserva todas las combinaciones disponibles;
- una **base histórica comparable**, con las unidades que tienen información en las seis elecciones;
- variables históricas descriptivas;
- un diagnóstico de cobertura para documentar las diferencias entre años.

> No se imputan valores históricos inexistentes. Si una categoría aparece solamente en un año, se conserva en la base maestra pero no se fuerza a tener una trayectoria de seis elecciones.


### Nota de homologación territorial

Antes de construir las trayectorias históricas se normalizan etiquetas territoriales equivalentes entre años. En particular, `DISTRITO FEDERAL` se homologa como `CIUDAD DE MEXICO` para evitar separar artificialmente una misma entidad por un cambio de nombre.


In [1]:
import os
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


## 1. Carga de las seis bases procesadas

El notebook busca primero los archivos en `/content/`. Si falta alguno, permite subir los CSV desde la computadora.

Archivos esperados:

- `conteos_censales_2009_agrupado.csv`
- `conteos_censales_2012_agrupado.csv`
- `conteos_censales_2015_agrupado.csv`
- `conteos_censales_2018_agrupado.csv`
- `conteos_censales_2021_agrupado.csv`
- `conteos_censales_2024_agrupado.csv`


In [2]:
ANIOS = [2009, 2012, 2015, 2018, 2021, 2024]

ARCHIVOS = {
    anio: f"/content/conteos_censales_{anio}_agrupado.csv"
    for anio in ANIOS
}

faltantes = [
    ruta for ruta in ARCHIVOS.values()
    if not os.path.exists(ruta)
]

if faltantes:
    print("Faltan archivos en /content/:")
    for ruta in faltantes:
        print(" -", os.path.basename(ruta))

    try:
        from google.colab import files
        print("\nSube los CSV procesados que falten.")
        files.upload()
    except ImportError:
        print("\nNo estás en Google Colab. Copia manualmente los archivos a /content/.")

faltantes = [
    ruta for ruta in ARCHIVOS.values()
    if not os.path.exists(ruta)
]

if faltantes:
    raise FileNotFoundError(
        "Todavía faltan archivos: "
        + ", ".join(os.path.basename(x) for x in faltantes)
    )

print("Las seis bases están disponibles.")


Las seis bases están disponibles.


## 2. Lectura y validación de estructura

Para la integración histórica solamente se exige que cada base contenga la llave común y los indicadores necesarios. Las demás columnas se conservan cuando son útiles para el diagnóstico.


In [3]:
LLAVES = [
    "EDONOM",
    "GRUPO_EDAD",
    "SEXO_DESC",
    "TIPOSEC_DESC"
]

OBLIGATORIAS = LLAVES + [
    "PARTICIPACION_PCT",
    "N_DETERMINADO",
    "GRUPO_PEQUENO"
]

bases = {}

for anio in ANIOS:
    ruta = ARCHIVOS[anio]
    df = pd.read_csv(ruta, low_memory=False)

    faltantes = [c for c in OBLIGATORIAS if c not in df.columns]
    if faltantes:
        raise ValueError(
            f"{anio}: faltan columnas obligatorias: {faltantes}"
        )

    # No debe haber más de una fila por unidad de análisis.
    duplicados = df.duplicated(LLAVES).sum()
    if duplicados != 0:
        raise ValueError(
            f"{anio}: se encontraron {duplicados} llaves duplicadas."
        )

    df["ANIO"] = anio
    bases[anio] = df

    print(
        f"{anio}: {len(df):,} grupos | "
        f"{df['EDONOM'].nunique()} entidades | "
        f"duplicados: {duplicados}"
    )


2009: 2,496 grupos | 32 entidades | duplicados: 0
2012: 2,496 grupos | 32 entidades | duplicados: 0
2015: 2,496 grupos | 32 entidades | duplicados: 0
2018: 2,496 grupos | 32 entidades | duplicados: 0
2021: 2,496 grupos | 32 entidades | duplicados: 0
2024: 2,546 grupos | 32 entidades | duplicados: 0


## 2.1 Homologación del nombre de Ciudad de México

El diagnóstico preliminar mostró una diferencia de nomenclatura territorial: en los procesos antiguos aparece `DISTRITO FEDERAL`, mientras que en los más recientes aparece `CIUDAD DE MEXICO`.

Estas etiquetas corresponden a la misma entidad federativa a través del periodo de estudio, por lo que se homologan a una sola categoría antes de integrar las bases:

`DISTRITO FEDERAL` → `CIUDAD DE MEXICO`

Esta corrección evita interpretar el cambio de nombre como si fueran unidades territoriales distintas.


In [4]:
# Homologación territorial necesaria para la serie histórica
for anio, df in bases.items():
    df["EDONOM"] = (
        df["EDONOM"]
        .astype(str)
        .str.strip()
        .str.upper()
        .replace({
            "DISTRITO FEDERAL": "CIUDAD DE MEXICO",
            "CIUDAD DE MÉXICO": "CIUDAD DE MEXICO"
        })
    )

    bases[anio] = df

print("Homologación territorial aplicada.")

# Verificación específica
for anio in ANIOS:
    etiquetas = sorted(
        bases[anio].loc[
            bases[anio]["EDONOM"].str.contains(
                "MEXICO|FEDERAL", case=False, na=False
            ),
            "EDONOM"
        ].unique().tolist()
    )
    print(anio, "->", etiquetas)


Homologación territorial aplicada.
2009 -> ['CIUDAD DE MEXICO', 'MEXICO']
2012 -> ['CIUDAD DE MEXICO', 'MEXICO']
2015 -> ['CIUDAD DE MEXICO', 'MEXICO']
2018 -> ['CIUDAD DE MEXICO', 'MEXICO']
2021 -> ['CIUDAD DE MEXICO', 'MEXICO']
2024 -> ['CIUDAD DE MEXICO', 'MEXICO']


## 3. Diagnóstico de categorías por año

Antes de unir las bases se revisan las categorías efectivamente disponibles. Esto es especialmente importante para evitar asumir que una categoría reciente existía en procesos anteriores.


In [5]:
for anio in ANIOS:
    df = bases[anio]

    print(f"\n--- {anio} ---")
    print("SEXO_DESC:")
    print(sorted(df["SEXO_DESC"].dropna().astype(str).unique().tolist()))

    print("TIPOSEC_DESC:")
    print(sorted(df["TIPOSEC_DESC"].dropna().astype(str).unique().tolist()))

    print("Grupos de edad:")
    print(sorted(df["GRUPO_EDAD"].dropna().astype(str).unique().tolist()))



--- 2009 ---
SEXO_DESC:
['Hombre', 'Mujer']
TIPOSEC_DESC:
['Mixta', 'Rural', 'Urbana']
Grupos de edad:
['18-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64', '65-69', '70-74', '75+']

--- 2012 ---
SEXO_DESC:
['Hombre', 'Mujer']
TIPOSEC_DESC:
['Mixta', 'Rural', 'Urbana']
Grupos de edad:
['18-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64', '65-69', '70-74', '75+']

--- 2015 ---
SEXO_DESC:
['Hombre', 'Mujer']
TIPOSEC_DESC:
['Mixta', 'Rural', 'Urbana']
Grupos de edad:
['18-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64', '65-69', '70-74', '75+']

--- 2018 ---
SEXO_DESC:
['Hombre', 'Mujer']
TIPOSEC_DESC:
['Mixta', 'Rural', 'Urbana']
Grupos de edad:
['18-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64', '65-69', '70-74', '75+']

--- 2021 ---
SEXO_DESC:
['Hombre', 'Mujer']
TIPOSEC_DESC:
['Mixta', 'Rural', 'Urbana']
Grupos de edad:
['18-19', '20-24

## 4. Base histórica en formato largo

Se concatenan los seis procesos manteniendo una fila por unidad y año. Esta versión facilita diagnósticos y gráficas por elección.


In [6]:
columnas_long = LLAVES + [
    "ANIO",
    "PARTICIPACION_PCT",
    "N_DETERMINADO",
    "GRUPO_PEQUENO"
]

# Conservar indicadores adicionales si están presentes en todas las bases.
opcionales = [
    "ABSTENCION_PCT",
    "NO_ESPECIFICADO_PCT",
    "LN", "SV", "NV", "NS"
]

for col in opcionales:
    if all(col in bases[a].columns for a in ANIOS):
        columnas_long.append(col)

df_long = pd.concat(
    [bases[a][columnas_long].copy() for a in ANIOS],
    ignore_index=True
)

print("Dimensiones de la base histórica larga:", df_long.shape)
print("Años presentes:", sorted(df_long["ANIO"].unique().tolist()))
display(df_long.head())


Dimensiones de la base histórica larga: (15026, 14)
Años presentes: [2009, 2012, 2015, 2018, 2021, 2024]


,EDONOM,GRUPO_EDAD,SEXO_DESC,TIPOSEC_DESC,ANIO,PARTICIPACION_PCT,N_DETERMINADO,GRUPO_PEQUENO,ABSTENCION_PCT,NO_ESPECIFICADO_PCT,LN,SV,NV,NS
0,AGUASCALIENTES,18-19,Hombre,Mixta,2009,36.785329,1854,False,63.214671,20.395019,2329,682,1172,475
1,AGUASCALIENTES,18-19,Hombre,Rural,2009,36.742757,2278,False,63.257243,0.000000,2278,837,1441,0
2,AGUASCALIENTES,18-19,Hombre,Urbana,2009,37.727104,10788,False,62.272896,13.613069,12488,4070,6718,1700
3,AGUASCALIENTES,18-19,Mujer,Mixta,2009,42.102665,2026,False,57.897335,20.549020,2550,853,1173,524
4,AGUASCALIENTES,18-19,Mujer,Rural,2009,49.355083,2636,False,50.644917,0.000000,2636,1301,1335,0


## 5. Cobertura histórica

Se cuenta en cuántas elecciones aparece cada combinación de entidad, grupo de edad, sexo y tipo de sección.


In [7]:
cobertura = (
    df_long.groupby(LLAVES, observed=True)["ANIO"]
    .nunique()
    .rename("ANIOS_DISPONIBLES")
    .reset_index()
)

print("Distribución del número de elecciones disponibles:")
display(
    cobertura["ANIOS_DISPONIBLES"]
    .value_counts()
    .sort_index()
    .rename_axis("ANIOS_DISPONIBLES")
    .to_frame("NUMERO_DE_UNIDADES")
)

print("\nUnidades con las seis elecciones:",
      (cobertura["ANIOS_DISPONIBLES"] == 6).sum())


Distribución del número de elecciones disponibles:


,NUMERO_DE_UNIDADES
ANIOS_DISPONIBLES,
1,50
6,2496



Unidades con las seis elecciones: 2496


## 6. Construcción de la base histórica ancha

Cada unidad queda en una sola fila y la participación de cada elección se almacena en una columna diferente.


In [8]:
participacion_wide = (
    df_long.pivot(
        index=LLAVES,
        columns="ANIO",
        values="PARTICIPACION_PCT"
    )
    .rename(columns={anio: f"PART_{anio}" for anio in ANIOS})
    .reset_index()
)

participacion_wide.columns.name = None

cols_part = [f"PART_{anio}" for anio in ANIOS]

participacion_wide["ANIOS_DISPONIBLES"] = (
    participacion_wide[cols_part].notna().sum(axis=1)
)

display(participacion_wide.head())
print("Dimensiones:", participacion_wide.shape)


,EDONOM,GRUPO_EDAD,SEXO_DESC,TIPOSEC_DESC,PART_2009,PART_2012,PART_2015,PART_2018,PART_2021,PART_2024,ANIOS_DISPONIBLES
0,AGUASCALIENTES,18-19,Hombre,Mixta,36.785329,51.249306,27.240223,47.432339,45.313640,52.481618,6
1,AGUASCALIENTES,18-19,Hombre,Rural,36.742757,54.276449,33.788820,51.050822,52.475667,51.201260,6
2,AGUASCALIENTES,18-19,Hombre,Urbana,37.727104,56.572840,31.144422,53.581582,42.256591,49.969146,6
3,AGUASCALIENTES,18-19,Mujer,Mixta,42.102665,58.915176,33.280934,55.596826,52.845828,60.411775,6
4,AGUASCALIENTES,18-19,Mujer,Rural,49.355083,65.429609,44.783307,61.058824,65.961692,64.677716,6


Dimensiones: (2546, 11)


## 7. Indicadores históricos derivados

Estos indicadores se generan principalmente para describir e interpretar los perfiles. No se usarán todos simultáneamente en K-means sin revisar previamente su correlación, ya que algunos resumen información contenida en las tasas por elección.


In [9]:
participacion_wide["PROMEDIO_HIST"] = (
    participacion_wide[cols_part].mean(axis=1)
)

participacion_wide["STD_HIST"] = (
    participacion_wide[cols_part].std(axis=1, ddof=0)
)

participacion_wide["PART_RECIENTE"] = participacion_wide["PART_2024"]

participacion_wide["CAMBIO_2009_2024"] = (
    participacion_wide["PART_2024"]
    - participacion_wide["PART_2009"]
)

def calcular_tendencia(row):
    xs = []
    ys = []

    for anio in ANIOS:
        valor = row[f"PART_{anio}"]
        if pd.notna(valor):
            xs.append(anio)
            ys.append(valor)

    if len(xs) < 2:
        return np.nan

    # Pendiente en puntos porcentuales por año.
    return np.polyfit(xs, ys, 1)[0]

participacion_wide["TENDENCIA_ANUAL"] = (
    participacion_wide.apply(calcular_tendencia, axis=1)
)

display(
    participacion_wide[
        LLAVES
        + cols_part
        + [
            "ANIOS_DISPONIBLES",
            "PROMEDIO_HIST",
            "STD_HIST",
            "CAMBIO_2009_2024",
            "TENDENCIA_ANUAL"
        ]
    ].head()
)


,EDONOM,GRUPO_EDAD,SEXO_DESC,TIPOSEC_DESC,PART_2009,PART_2012,PART_2015,PART_2018,PART_2021,PART_2024,ANIOS_DISPONIBLES,PROMEDIO_HIST,STD_HIST,CAMBIO_2009_2024,TENDENCIA_ANUAL
0,AGUASCALIENTES,18-19,Hombre,Mixta,36.785329,51.249306,27.240223,47.432339,45.313640,52.481618,6,43.417076,8.840246,15.696289,0.770158
1,AGUASCALIENTES,18-19,Hombre,Rural,36.742757,54.276449,33.788820,51.050822,52.475667,51.201260,6,46.589296,8.121203,14.458504,0.801449
2,AGUASCALIENTES,18-19,Hombre,Urbana,37.727104,56.572840,31.144422,53.581582,42.256591,49.969146,6,45.208614,8.985208,12.242042,0.387606
3,AGUASCALIENTES,18-19,Mujer,Mixta,42.102665,58.915176,33.280934,55.596826,52.845828,60.411775,6,50.525534,9.725795,18.309110,0.910985
4,AGUASCALIENTES,18-19,Mujer,Rural,49.355083,65.429609,44.783307,61.058824,65.961692,64.677716,6,58.544372,8.368328,15.322633,0.899856


## 8. Control histórico de grupos pequeños

Se marca si una unidad presentó `GRUPO_PEQUENO=True` en al menos una de las elecciones disponibles. Esto permitirá hacer análisis de sensibilidad en el clustering.


In [10]:
pequenos = (
    df_long.groupby(LLAVES, observed=True)["GRUPO_PEQUENO"]
    .max()
    .rename("ALGUN_GRUPO_PEQUENO")
    .reset_index()
)

base_maestra = participacion_wide.merge(
    pequenos,
    on=LLAVES,
    how="left",
    validate="one_to_one"
)

print("ALGUN_GRUPO_PEQUENO:")
display(base_maestra["ALGUN_GRUPO_PEQUENO"].value_counts(dropna=False))


ALGUN_GRUPO_PEQUENO:


,count
ALGUN_GRUPO_PEQUENO,
False,2470
True,76


## 9. Base comparable de seis elecciones

La muestra principal para la segmentación histórica estará formada por las unidades que cuentan con información en las seis elecciones.

Este criterio evita imputar artificialmente años faltantes. Las unidades con menor cobertura se mantienen en la base maestra para análisis descriptivo.


In [11]:
base_comparable = base_maestra.loc[
    base_maestra["ANIOS_DISPONIBLES"] == 6
].copy()

print("Unidades en la base maestra:", len(base_maestra))
print("Unidades con trayectoria completa:", len(base_comparable))
print(
    "Porcentaje con trayectoria completa:",
    round(len(base_comparable) / len(base_maestra) * 100, 2)
)

print("\nCategorías de sexo en la base comparable:")
display(base_comparable["SEXO_DESC"].value_counts(dropna=False))

print("\nTipos de sección en la base comparable:")
display(base_comparable["TIPOSEC_DESC"].value_counts(dropna=False))


Unidades en la base maestra: 2546
Unidades con trayectoria completa: 2496
Porcentaje con trayectoria completa: 98.04

Categorías de sexo en la base comparable:


,count
SEXO_DESC,
Hombre,1248
Mujer,1248



Tipos de sección en la base comparable:


,count
TIPOSEC_DESC,
Mixta,832
Rural,832
Urbana,832


## 10. Validaciones finales de integración

Antes de exportar, se verifica:

- que no existan llaves duplicadas;
- que la base comparable tenga las seis tasas de participación completas;
- que los porcentajes estén dentro del intervalo 0--100.


In [12]:
duplicados_maestra = base_maestra.duplicated(LLAVES).sum()
duplicados_comparable = base_comparable.duplicated(LLAVES).sum()

faltantes_comparable = base_comparable[cols_part].isna().sum().sum()

fuera_rango = (
    ((base_comparable[cols_part] < 0) |
     (base_comparable[cols_part] > 100))
    .sum()
    .sum()
)

print("Duplicados en base maestra:", duplicados_maestra)
print("Duplicados en base comparable:", duplicados_comparable)
print("Valores de participación faltantes en comparable:", faltantes_comparable)
print("Valores de participación fuera de 0-100:", fuera_rango)

if duplicados_maestra != 0 or duplicados_comparable != 0:
    raise ValueError("Hay llaves duplicadas en la integración.")

if faltantes_comparable != 0:
    raise ValueError("La base comparable contiene tasas faltantes.")

if fuera_rango != 0:
    raise ValueError("Hay tasas de participación fuera del rango 0-100.")

print("\nValidación final de integración: OK")


Duplicados en base maestra: 0
Duplicados en base comparable: 0
Valores de participación faltantes en comparable: 0
Valores de participación fuera de 0-100: 0

Validación final de integración: OK


## 11. Exportación

Se generan tres productos:

1. `base_historica_2009_2024.csv`: todas las unidades disponibles.
2. `base_historica_comparable_2009_2024.csv`: unidades con seis elecciones.
3. `diagnostico_cobertura_historica.csv`: número de elecciones disponibles por unidad.


In [13]:
SALIDA_MAESTRA = "/content/base_historica_2009_2024.csv"
SALIDA_COMPARABLE = "/content/base_historica_comparable_2009_2024.csv"
SALIDA_COBERTURA = "/content/diagnostico_cobertura_historica.csv"

base_maestra.to_csv(
    SALIDA_MAESTRA,
    index=False,
    encoding="utf-8-sig"
)

base_comparable.to_csv(
    SALIDA_COMPARABLE,
    index=False,
    encoding="utf-8-sig"
)

cobertura.to_csv(
    SALIDA_COBERTURA,
    index=False,
    encoding="utf-8-sig"
)

print("Archivos generados:")
print(" -", SALIDA_MAESTRA)
print(" -", SALIDA_COMPARABLE)
print(" -", SALIDA_COBERTURA)


Archivos generados:
 - /content/base_historica_2009_2024.csv
 - /content/base_historica_comparable_2009_2024.csv
 - /content/diagnostico_cobertura_historica.csv


# Notas para el documento técnico

La integración histórica se realiza mediante una llave común formada por entidad federativa, grupo de edad, sexo y tipo de sección. Antes de la unión se homologó la etiqueta `DISTRITO FEDERAL` como `CIUDAD DE MEXICO` para mantener la continuidad territorial de la serie. Se conserva una base maestra con todas las combinaciones observadas y se construye una segunda base comparable con las unidades que cuentan con información en las seis elecciones.

No se imputan artificialmente valores para categorías ausentes en ciertos años. Las tasas de participación de 2009, 2012, 2015, 2018, 2021 y 2024 se mantienen como variables históricas principales. Los indicadores de promedio, variabilidad, cambio y tendencia se calculan como variables descriptivas y su uso en el clustering se decidirá después de revisar la correlación entre variables.

El siguiente paso será realizar el análisis exploratorio de la base comparable, estandarizar las variables seleccionadas y evaluar soluciones de K-means con cuatro, cinco y seis conglomerados.
